In [ ]:
!pip install icrawler

In [ ]:
# Step 1: Try unmounting (ignore error if any)
!fusermount -u /content/drive

# Step 2: Delete ghost mount folder
!rm -rf /content/drive

# Step 3: Now mount cleanly
from google.colab import drive
drive.mount('/content/drive')

fusermount: failed to unmount /content/drive: No such file or directory
Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from icrawler.builtin import GoogleImageCrawler
import os
#Junk Filter
#-art -illustration -AI -generated -wallpaper -vector -sticker -tattoo -poster -toy -decoration
#List of Plants
#['pothos plant', 'snake plant', 'spider plant', 'orchid', 'cathedral cactus', 'peperomia prostrata']
#Pothos Diseases
#Fungal Root Rots, Southern Blight Anthracnose/leaf spot, Botrytis (Gray Mold), Mealy Bugs, Soft Scale, Spider Mites, Over Watering


# Folder in your Google Drive where images will be saved
base_path = '/content/drive/MyDrive/MLPlant'

# List of plant species to scrape
species_list = ['pothos plant', 'snake plant', 'spider plant', 'orchid', 'cathedral cactus', 'peperomia prostrata']

for species in species_list:
    species_folder = os.path.join(base_path, species.replace(" ", "_"))
    os.makedirs(species_folder, exist_ok=True)

    crawler = GoogleImageCrawler(storage={'root_dir': species_folder})
    crawler.crawl(keyword=species, max_num=100)

Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable


KeyboardInterrupt: 

In [ ]:
general_modifiers = [
    "indoor houseplant photo",
    "full plant in pot",
    "whole plant indoors",
    "plant on windowsill",
    "single plant in pot",
    "leaf close-up",
    "leaf macro photo",
    "leaf underside close-up",
    "leaf detail with veins visible",
    "natural indoor lighting",
    "soft indoor light",
    "against plain background",
    "against white background",
    "against neutral background",
    "top-down view",
    "side view of plant",
    "plant on table indoors",
]

#condition specific modifiers for each of species_list entries
condition_modifiers = {
    "pothos healthy": [
        "healthy foliage close-up",
        "leaf detail with no damage",
        "lush green leaves",
        "vigorous growth",
        "full plant with healthy leaves",
        "new growth close-up",
        "healthy vine hanging from pot",
        "healthy leaves with no spots",
    ],

    "pothos root rot": [
        "roots removed from pot",
        "root ball close-up",
        "roots and soil visible",
        "stem base at soil line",
        "mushy roots close-up",
        "darkened roots near crown",
        "plant wilting in wet soil",
        "base of stem showing decay",
    ],

    "pothos leaf spot": [
        "leaf with brown spots",
        "macro of leaf lesions",
        "leaf margin detail",
        "leaf with irregular spots",
        "leaf showing necrotic patches",
        "leaf spots with yellow halos",
        "close-up of damaged leaf surface",
        "cluster of leaves with spots",
    ],

    "pothos mealybugs": [
        "cottony pests on leaf nodes",
        "white fuzzy insects on stems",
        "pests clustered at leaf axils",
        "macro of insects on leaves",
        "pests on leaf underside",
        "close-up of cottony clusters",
        "honeydew and pests on foliage",
        "infested node close-up",
    ],

    "pothos scale insects": [
        "brown bumps along leaf midrib",
        "pests attached to stems",
        "small dome-shaped insects on leaves",
        "close-up of insects on stem",
        "pests along veins of leaf",
        "honeydew and sooty mold on leaves",
        "cluster of bumps on foliage",
        "infested stem section close-up",
    ],

    "pothos spider mites": [
        "fine webbing on leaf underside",
        "leaf with stippling damage",
        "speckled leaf surface close-up",
        "webbing between leaf and stem",
        "pale spots scattered on leaf",
        "macro of tiny pests on leaf",
        "damaged foliage with webbing",
        "close-up of underside with webbing",
    ],

    "pothos edema": [
        "leaf with water blisters",
        "translucent patches on leaf surface",
        "raised lesions on foliage",
        "close-up of swollen leaf tissue",
        "blistered areas between veins",
        "patchy translucent spots on leaf",
        "lower leaf surface with bumps",
        "macro of textured leaf lesions",
    ],
}

In [ ]:
from icrawler import ImageDownloader
metadata = []

class LoggingDownloader(ImageDownloader):
    def __init__(self, *args, **kwargs):
        # pull our extra info out of kwargs
        self.condition = kwargs.pop('condition')
        self.search_term = kwargs.pop('search_term')
        self.meta_list = kwargs.pop('meta_list')
        super().__init__(*args, **kwargs)

    def download(self, task, default_ext, timeout=5, **kwargs):
        # Let the parent class actually download the file
        result = super().download(task, default_ext, timeout=timeout, **kwargs)

        if result:
            #self.storage is set up by icrawler; get the actual saved path
            filename = self.get_filename(task, default_ext)
            file_path = os.path.join(self.storage['root_dir'], filename)

            #log one row of metadata
            self.meta_list.append({
                'filename': filename,
                'filepath': file_path,
                'condition': self.condition,
                'search_term': self.search_term,
                'url': task['file_url'],
            })

        return result

In [ ]:
#from icrawler.builtin import GoogleImageCrawler
from icrawler.builtin import BingImageCrawler
import os, random
#turn off negatives

base_path = '/content/drive/MyDrive/MLPlant'

condition_list = ['pothos healthy',
                'pothos root rot',
                'pothos leaf spot',
                'pothos mealybugs',
                'pothos scale insects',
                'pothos spider mites',
                'pothos edema',]
negative_terms = [
    "art", "illustration", "AI", "generated", "wallpaper",
    "vector", "sticker", "tattoo", "poster", "toy", "decoration"
]
negative_suffix = " " + " ".join(f"-{t}" for t in negative_terms)

metadata = [] #store meta data in here add to csv at end

#Main loop: goes through the list of conditions
for i in condition_list:
    search_terms = [] #resets collected search terms per condition

    condition_folder = os.path.join(base_path, i.replace(" ", "_")) #makes new folder name for each condition inside of file path
    os.makedirs(condition_folder, exist_ok=True) # makes the actual folder
    special_modifiers = condition_modifiers.get(i) # queries python dictionary of special modifiers


    if special_modifiers is not None: #makes sure query didnt fail or is empty
      for k in special_modifiers: #loops through the dictionary entry adding the list of negaive suffixes
        #search_term = i + " " + k
        #search_terms.append(search_term)
        search_terms.append(i + " " + k + negative_suffix) #the condition + special modifier + negative search terms
        #end up with something like pothos root rot + root ball close up + the negatives

    for j in general_modifiers: #loops through general modifiers
      #search_term = i + " " + j
      #print(search_term)
      #search_terms.append(search_term)
      search_terms.append(i + " " + j + negative_suffix) # adds condition + general modifier + negative search terms

    random.shuffle(search_terms)
    search_terms = search_terms[:10] #grab a random set of 10 search terms from search term list
    print(search_terms)
    print(len(search_terms))


    for search_term in search_terms: #loop through our 10 search terms
      print(search_term)
      crawler = BingImageCrawler(storage={'root_dir': condition_folder}) # create the instance of our crawler
      crawler.crawl(keyword=search_term, max_num=100) # tell it what to look for (the search term)and the max amount we want of it
      metadata.append({
            'condition': i,
            'search_term': search_term, #grab the condition the search term and were we stored it
            'folder': condition_folder,
        })
#saves meta data into a csv file
meta_df = pd.DataFrame(metadata)
meta_csv_path = os.path.join(base_path, 'pothos_search_metadata.csv')
meta_df.to_csv(meta_csv_path, index=False)
print("Saved metadata to:", meta_csv_path)





['pothos healthy leaf detail with veins visible', 'pothos healthy healthy vine hanging from pot', 'pothos healthy top-down view', 'pothos healthy leaf detail with no damage', 'pothos healthy single plant in pot', 'pothos healthy soft indoor light', 'pothos healthy full plant with healthy leaves', 'pothos healthy against white background', 'pothos healthy healthy leaves with no spots', 'pothos healthy lush green leaves']
10
pothos healthy leaf detail with veins visible


ERROR:downloader:Response status code 404, file https://positivebloom.com/wp-content/uploads/2023/01/golden-pothos-leaf-with-yellow-details.jpg
ERROR:downloader:Response status code 403, file https://preview.redd.it/golden-pothos-v0-z56jjv6k6ula1.jpg


pothos healthy healthy vine hanging from pot


ERROR:downloader:Response status code 403, file https://preview.redd.it/golden-pothos-v0-z56jjv6k6ula1.jpg


pothos healthy top-down view


...

pothos edema natural indoor lighting


ERROR:downloader:Response status code 403, file https://preview.redd.it/golden-pothos-v0-z56jjv6k6ula1.jpg


pothos edema lower leaf surface with bumps


ERROR:downloader:Response status code 403, file https://preview.redd.it/golden-pothos-v0-z56jjv6k6ula1.jpg


Saved metadata to: /content/drive/MyDrive/MLPlant/pothos_search_metadata.csv


In [ ]:
import os
import hashlib

def remove_duplicates(base_path, folders, dry_run=True):

    # include .webp now
    valid_exts = ('.jpg', '.jpeg', '.png', '.webp')

    seen_hashes = {}     # hash -> first file path seen
    removed_files = []   # list of files that are (or would be) removed

    for folder in folders: # loop through list of folders
        folder_path = os.path.join(base_path, folder)
        files = [
            f for f in os.listdir(folder_path) #gets the files from the folders
            if f.lower().endswith(valid_exts)
        ]
        for fname in files: #loops through files per folder
            fpath = os.path.join(folder_path, fname)
            # read file and compute md5 hash
            with open(fpath, 'rb') as f: #opens file reads hashmap
                file_hash = hashlib.md5(f.read()).hexdigest()

            if file_hash in seen_hashes: #if the has is the same as one in seen hases
                # duplicate found
                if dry_run:
                    print(f"[DRY RUN] Would remove duplicate: {fpath}") #print if dry run
                else:
                    os.remove(fpath)
                    print(f"Removed duplicate: {fpath}") #delete if real
                removed_files.append(fpath)
            else:
                # first time seeing this hash
                seen_hashes[file_hash] = fpath

    if dry_run:
        print(f"\n[DRY RUN] Found {len(removed_files)} duplicates. No files were deleted.")
    else:
        print(f"\nRemoved {len(removed_files)} duplicate files.")

    return removed_files


Rerun set base_path and import os

In [ ]:
import os
base_path = '/content/drive/MyDrive/MLPLANT3'
print(os.listdir(base_path))
oriented_folders = ['pothos_healthy', 'pothos_leaf_spot', 'pothos_pests', 'pothos_root_rot',]

['pothos_healthy', 'pothos_leaf_spot', 'pothos_pests', 'pothos_root_rot']


In [ ]:

removed_files_preview = remove_duplicates(base_path, oriented_folders, dry_run=True)

[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot079.jpg
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot081.webp
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot082.webp
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot093.jpg
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot097.jpg
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot098.jpg
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot102.webp
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_pests/pothos_pests043.webp
[DRY RUN] Would remove duplicate: /content/drive/MyDrive/MLPLANT3/pothos_pests/pothos_pests044.webp
[DRY RUN] Would remove duplicate: /content/drive

In [ ]:
removed_files_preview = remove_duplicates(base_path, oriented_folders, dry_run=False) #actually get rid of duplicates

Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot079.jpg
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot081.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot082.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot093.jpg
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot097.jpg
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot098.jpg
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_leaf_spot/pothos_leaf_spot102.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_pests/pothos_pests043.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_pests/pothos_pests044.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_pests/pothos_pests060.webp
Removed duplicate: /content/drive/MyDrive/MLPLANT3/pothos_root_rot/pothos_root_rot008.jpeg
Removed

In [ ]:
removed_files_preview = remove_duplicates(base_path, oriented_folders, dry_run=True) #check


[DRY RUN] Found 0 duplicates. No files were deleted.


In [ ]:
import shutil
import os
#make back up before image editing
src = '/content/drive/MyDrive/MLPLANT3'

dst = '/content/drive/MyDrive/MLPLANT3_backup'

if os.path.exists(dst):
    print("Backup folder already exists — skipping copy.")
else:
    shutil.copytree(src, dst)
    print("Backup of MLPLANT3 created successfully at:", dst)

Backup of MLPLANT3 created successfully at: /content/drive/MyDrive/MLPLANT3_backup


Image Variance

In [ ]:
import random
from PIL import Image, ImageEnhance


# New folder where augmented dataset will go
output_root = base_path + "_aug"
os.makedirs(output_root, exist_ok=True)

classes = [
    "pothos_healthy",
    "pothos_leaf_spot",
    "pothos_pests",
    "pothos_root_rot",
]


def augment_image(img):

    aug_images = []

    # flip
    aug_images.append(img.transpose(Image.FLIP_LEFT_RIGHT))

    #small random rotation -20 to +20 degrees
    angle = random.uniform(-20, 20)
    aug_images.append(img.rotate(angle, resample=Image.BICUBIC, expand=True))

    # random crop/zoom 80–100% of original then resize back
    w, h = img.size
    scale = random.uniform(0.8, 1.0)  # 80–100% crop
    new_w, new_h = int(w * scale), int(h * scale)
    left = random.randint(0, w - new_w)
    top = random.randint(0, h - new_h)
    crop = img.crop((left, top, left + new_w, top + new_h))
    aug_images.append(crop.resize((w, h), Image.BICUBIC))

    # brightness/contrast jitter +20% -20%
    b_factor = random.uniform(0.8, 1.2)
    c_factor = random.uniform(0.8, 1.2)
    bright = ImageEnhance.Brightness(img).enhance(b_factor)
    contrast = ImageEnhance.Contrast(bright).enhance(c_factor)
    aug_images.append(contrast)

    return aug_images


for cls in classes:
    input_folder = os.path.join(base_path, cls)
    output_folder = os.path.join(output_root, cls)
    os.makedirs(output_folder, exist_ok=True)

    print(f"Processing class: {cls}")

    for fname in os.listdir(input_folder): #loops through file names
        if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): #if file name isnt one of these it skips
            continue

        in_path = os.path.join(input_folder, fname)
        # make RGB
        img = Image.open(in_path).convert("RGB")

        base_name, ext = os.path.splitext(fname)

        #save og copy
        orig_out = os.path.join(output_folder, f"{base_name}_orig{ext}")
        img.save(orig_out)

        # save change versions
        for idx, aug_img in enumerate(augment_image(img), start=1):
            out_path = os.path.join(output_folder, f"{base_name}_aug{idx}{ext}")
            aug_img.save(out_path)

print("Augmented dataset savedin:", output_root)


Processing class: pothos_healthy
Processing class: pothos_leaf_spot
Processing class: pothos_pests
Processing class: pothos_root_rot
Augmented dataset savedin: /content/drive/MyDrive/MLPLANT3_aug


In [ ]:
import cv2
import os
import numpy as np

base_path = '/content/drive/MyDrive/MLPLANT3_aug'
base_output_path = base_path + "_cropped224"
os.makedirs(base_output_path, exist_ok=True)
plant_folders = ["pothos_healthy", "pothos_leaf_spot", "pothos_pests", "pothos_root_rot",]

# rezive and add padding
def resize_and_pad(img, size=(224, 224)):
    h, w = img.shape[:2]
    scale = min(size[0] / h, size[1] / w)
    nh, nw = int(h * scale), int(w * scale)
    img_resized = cv2.resize(img, (nw, nh))

    # create new image and center inside
    result = np.zeros((size[0], size[1], 3), dtype=np.uint8)
    y_offset = (size[0] - nh)// 2
    x_offset = (size[1] - nw) // 2
    result[y_offset:y_offset+nh, x_offset:x_offset+nw] = img_resized

    return result

for plant_folder in plant_folders:
    input_folder = os.path.join(base_path, plant_folder)
    output_folder = os.path.join(base_output_path, plant_folder)
    os.makedirs(output_folder, exist_ok=True)

    print(f"Processing folder:{plant_folder}")

    for filename in os.listdir(input_folder):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue  # openCV cant handle avif files, just leave them alone they wont be in completeed dataset

        path = os.path.join(input_folder, filename)
        img = cv2.imread(path)
        if img is None:
            print(f"  Skipping unreadable file: {filename}")
            continue

        # Convert to grayscale blur and threshold
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(
            blurred, 50, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        # find contours assume largest is main plant/leaf area
        contours, _ = cv2.findContours(
            thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours:
            print(f" No contours found, skipping: {filename}")
            continue

        best_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(best_contour)

        if w < 50 or h < 50 or w/h > 4 or h/w > 4:
            print(f"  Weird bounding box, skipping: {filename}")
            continue

        # Add padding around the crop
        pad = int(max(w, h) * 0.2)
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img.shape[1], x + w + pad)
        y2 = min(img.shape[0], y + h + pad)
        cropped = img[y1:y2, x1:x2]
        resized = resize_and_pad(cropped, size=(224, 224))

        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, resized)

    print(f"Finished processing {plant_folder}")

print("Cropped dataset at:", base_output_path)


Processing folder:pothos_healthy
Finished processing pothos_healthy
Processing folder:pothos_leaf_spot
  Weird bounding box, skipping: pothos_leaf_spot009_aug3.jpg
Finished processing pothos_leaf_spot
Processing folder:pothos_pests
  Weird bounding box, skipping: pothos_pests068_orig.jpg
  Weird bounding box, skipping: pothos_pests068_aug1.jpg
  Weird bounding box, skipping: pothos_pests068_aug3.jpg
Finished processing pothos_pests
Processing folder:pothos_root_rot
  Weird bounding box, skipping: pothos_root_rot090_orig.jpg
  Weird bounding box, skipping: pothos_root_rot090_aug1.jpg
  Weird bounding box, skipping: pothos_root_rot090_aug3.jpg
  Weird bounding box, skipping: pothos_root_rot090_aug4.jpg
Finished processing pothos_root_rot
Cropped dataset at: /content/drive/MyDrive/MLPLANT3_aug_cropped224


SPlit

In [ ]:
import os
import shutil
import random

# AUGMENTED + CROPPED224 dataset
input_folder = '/content/drive/MyDrive/MLPLANT3_aug_cropped224'

output_base = input_folder + "_split"   # /content/drive/MyDrive/MLPLANT3_aug_cropped224_split

splits = ['train', 'val', 'test']

plant_folders = [
    "pothos_healthy",
    "pothos_leaf_spot",
    "pothos_pests",
    "pothos_root_rot",
]

# make train val and test folders
for split in splits:
    for plant in plant_folders:
        os.makedirs(os.path.join(output_base, split, plant), exist_ok=True)

# Split ratios
train_ratio = 0.7
val_ratio   = 0.15
test_ratio  = 0.15

# split seed
random.seed(42)

#split images for each class
for plant in plant_folders:
    species_path = os.path.join(input_folder, plant)
    images = [
        f for f in os.listdir(species_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
    ]

    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * train_ratio)
    n_val   = int(n_total * val_ratio)

    train_files = images[:n_train]
    val_files   = images[n_train:n_train + n_val]
    test_files  = images[n_train + n_val:]

    #copy train files
    for img_file in train_files:
        shutil.copy(
            os.path.join(species_path, img_file),
            os.path.join(output_base, 'train', plant, img_file)
        )

    #copy val files
    for img_file in val_files:
        shutil.copy(
            os.path.join(species_path, img_file),
            os.path.join(output_base, 'val', plant, img_file)
        )

    # copy test files
    for img_file in test_files:
        shutil.copy(
            os.path.join(species_path, img_file),
            os.path.join(output_base, 'test', plant, img_file)
        )

    print(f"{plant}: {len(train_files)} train, {len(val_files)} val, {len(test_files)} test")

print("Finished splitting dataset into train/val/test")
print("Split dataset at:", output_base)


pothos_healthy: 465 train, 99 val, 101 test
pothos_leaf_spot: 391 train, 83 val, 85 test
pothos_pests: 288 train, 61 val, 63 test
pothos_root_rot: 333 train, 71 val, 72 test
Finished splitting dataset into train/val/test
Split dataset at: /content/drive/MyDrive/MLPLANT3_aug_cropped224_split
